# Replay And Comparative Walkthrough

Phase 6 Increment 3 notebook for reproducible replay/comparison analysis.

Workflow:
1. Optionally regenerate canonical multi-seed artifacts.
2. Load the latest multi-seed report.
3. Build a pairwise comparative report from two run directories.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys

def resolve_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not resolve repository root from current working directory")

ROOT = resolve_repo_root(Path.cwd())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

ROOT


In [ ]:
RUN_GENERATORS = False

if RUN_GENERATORS:
    env = dict(os.environ)
    env["PYTHONPATH"] = "src"
    commands = [
        [
            sys.executable,
            "scripts/run_multi_seed_report.py",
            "--config",
            "configs/experiments/multi_seed_baseline.json",
        ],
    ]
    for command in commands:
        print("RUN", " ".join(command))
        subprocess.run(command, cwd=ROOT, env=env, check=True)
else:
    print("RUN_GENERATORS is False; using existing artifacts")


In [ ]:
report_candidates = sorted(
    (ROOT / "artifacts" / "reports").glob("multi_seed_report_*.json"),
    key=lambda path: path.stat().st_mtime,
)
if not report_candidates:
    raise FileNotFoundError("No multi-seed report found. Generate one with scripts/run_multi_seed_report.py")

multi_seed_report_path = report_candidates[-1]
multi_seed_report = json.loads(multi_seed_report_path.read_text(encoding="utf-8"))
runs = multi_seed_report.get("runs", [])
if len(runs) < 2:
    raise RuntimeError("Need at least two runs in the selected multi-seed report")

left_run_dir = Path(runs[0]["run_dir"])
right_run_dir = Path(runs[1]["run_dir"])

multi_seed_report_path, left_run_dir, right_run_dir


In [ ]:
from visualization import generate_comparative_report

comparison_output = generate_comparative_report(
    left_run_dir=left_run_dir,
    right_run_dir=right_run_dir,
    reports_root=ROOT / "artifacts" / "reports",
    figures_root=ROOT / "artifacts" / "figures",
)

comparison_output["report_file"], comparison_output["summary_file"]


In [ ]:
report = comparison_output["report"]
overview = {
    "left_run_id": report["left_run"]["run_id"],
    "right_run_id": report["right_run"]["run_id"],
    "final_compromised_nodes_delta": report["comparisons"]["final_compromised_nodes_delta"],
    "blue_containment_actions_delta": report["comparisons"]["blue_containment_actions_delta"],
    "response_latency_delta": report["comparisons"]["response_latency_delta"],
    "sequence_hash_match": report["comparisons"]["sequence_hash_match"],
}
overview
